# Case 4 — Multi-Rate Variable Production
## POD Analysis: Rate-Transient Diagnostics, Reduced-Order Model

Case 4 uses the same homogeneous reservoir as Case 1 but applies a four-period variable-rate schedule, enabling Rate-Transient Analysis (RTA).

| Parameter | Case 1 | Case 4 |
|-----------|--------|--------|
| $k$ | 20 mD | 20 mD (same) |
| Rate | constant 800 STB/day | variable: 800→400→1200→0 |
| Schedule | 350 log steps | 4 periods, 350 steps total |
| Analysis | PTA | RTA |


In [ ]:
import numpy as np
import h5py, json
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import UnivariateSpline
from matplotlib.ticker import LogLocator, LogFormatter
from scipy.special import expi
from pathlib import Path
import matplotlib.patches as patches
import matplotlib.ticker as ticker
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
import warnings
from matplotlib.ticker import NullLocator
import matplotlib.lines as mlines

plt.rcParams.update({
    # 1. True LaTeX Font Rendering
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'], # Native LaTeX font
    
    # Optional
    'text.latex.preamble': r'\usepackage{amsmath} \usepackage{amssymb} \usepackage{bm}',

    # 2. Font Sizes 
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 14,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,

    # 3. Ticks: Inward-facing and mirrored on top/right
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': False,
    'ytick.minor.visible': False,

    # 4. Line and Spine Weights
    'axes.linewidth': 1,      # Frame thickness
    'lines.linewidth': 1.5,     # Data line thickness
    'lines.markersize': 6,      # Default marker size

    # 5. Grid (Keep it subtle if used)
    'axes.grid': False,
    'grid.alpha': 0.20,
    'grid.linestyle': '--',
    'grid.linewidth': 0.5,

    # 6. Saving and DPI
    'figure.dpi': 150,          # For crisp screen preview in Jupyter/Spyder
    'savefig.dpi': 600,         # High resolution for line art
    'savefig.bbox': 'tight',    # Prevents labels from getting cut off

    # legend
    # 7. Legend Formatting
    'legend.frameon': True,         
    'legend.framealpha': 1.0,       
    'legend.edgecolor': 'black',    
    'legend.fancybox': False,       
    'legend.handlelength': 1.5,     
    'legend.labelspacing': 0.4,     
    'legend.handletextpad': 0.5,    
    'legend.borderaxespad': 0.5,   

    # axes padding
    'xtick.major.pad': 5.0,  # Default is usually 3.5
    'ytick.major.pad': 5.0, 
})

C1,C2,C3,C4 = '#1a5e9e','#0d6e56','#9b2a1a','#b07318'
GR = '#555555'
# Period colours
PC = ['#1a5e9e','#b07318','#0d6e56','#9b2a1a']  # P1 blue, P2 amber, P3 teal, P4 red

PROJECT = Path('..')
DATA    = PROJECT/'data'/'01_synthetic'/'case4_variable_rate'
FIGS = PROJECT/'figures'
FIGS.mkdir(parents=True, exist_ok=True)
def sf(n): plt.savefig(FIGS/n,dpi=300,bbox_inches='tight'); print(f'  {n}')
print('Ready.')

---
## 1 · Load Simulation Data

Case 4 exports `rate_STBd` as an array (not a scalar) and `cum_prod_STB` for material balance calculations. `period_start_hr` defines the boundary between rate periods for regime masking.

In [ ]:
with h5py.File(DATA/'snapshots_psia.mat','r') as f:
    _raw   = np.array(f['X_psia'])
    X_psia = _raw.T if _raw.shape[0]<_raw.shape[1] else _raw

with h5py.File(DATA/'well_data.mat','r') as f:
    bhp_raw  = np.array(f['bhp_psia']).flatten()
    dp_raw   = np.array(f['delta_p_psi']).flatten()
    q_raw    = np.array(f['rate_STBd']).flatten()
    t_raw    = np.array(f['time_hr']).flatten()
    Np_raw   = np.array(f['cum_prod_STB']).flatten()

with h5py.File(DATA/'rock_field.mat','r') as f:
    perm_mD = np.array(f['perm_mD']).flatten()

with open(DATA/'params.json') as f:
    P = json.load(f)

n_t     = min(len(t_raw), X_psia.shape[1])
t_raw   = t_raw[:n_t]; bhp_raw = bhp_raw[:n_t]
dp_raw  = dp_raw[:n_t]; q_raw  = q_raw[:n_t]
Np_raw  = Np_raw[:n_t]; X_psia = X_psia[:,:n_t]

pi_   = P['p_init_psia']
valid = (t_raw>0)&(dp_raw>0)&(dp_raw<pi_)&(bhp_raw>0)&(bhp_raw<=pi_)
t   = t_raw[valid];  dp  = dp_raw[valid]
bhp = bhp_raw[valid]; q  = q_raw[valid]
Np  = Np_raw[valid];  X  = X_psia[:,valid]

N, m = X.shape
nx   = int(P['nx']); ny = int(P['ny'])
wc   = int(P['well_cell_matlab']) - 1

# Period boundaries
t_wbs   = P['t_wbs_end_hr']
t_pss   = P['t_pss_start_hr']
t_p1end = P['t_period1_end_hr']   # 1000 hr
t_p2end = P['t_period2_end_hr']   # 2500 hr
t_p3end = P['t_period3_end_hr']   # 4000 hr
t_shut  = P['t_shutin_hr']        # 4000 hr

# Period masks
m_p1  = (t >  0)     & (t <= t_p1end)
m_p2  = (t >  t_p1end) & (t <= t_p2end)
m_p3  = (t >  t_p2end) & (t <= t_p3end)
m_bu  = (t >  t_shut)

# WBS and PSS masks within Period 1
m_wbs = (t >= 0)    & (t < t_wbs)  & m_p1
m_rf  = (t >= t_wbs)& (t < t_pss)  & m_p1
m_bdf = (t >= t_pss)& m_p1

print(f'X shape          : {X.shape}')
print(f'Time range       : {t[0]:.5f} – {t[-1]:.0f} hr')
print(f'BHP range        : {bhp.min():.1f} – {bhp.max():.1f} psia')
print(f'Rate range       : {q.min():.0f} – {q.max():.0f} STB/day')
print(f'Cum. prod        : {Np.max():.0f} STB')
print(f'Period 1 steps   : {m_p1.sum()} (q={P["q1_STBd"]:.0f} STB/day)')
print(f'Period 2 steps   : {m_p2.sum()} (q={P["q2_STBd"]:.0f} STB/day)')
print(f'Period 3 steps   : {m_p3.sum()} (q={P["q3_STBd"]:.0f} STB/day)')
print(f'Buildup  steps   : {m_bu.sum()}')
print(f't_WBS={t_wbs:.2f} hr  t_PSS={t_pss:.1f} hr')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

# Panel (a): rate history
ax = axes[0]
for mask, col, lbl in [(m_p1,PC[0],'P1: $q=800$'),(m_p2,PC[1],'P2: $q=400$'),
                        (m_p3,PC[2],'P3: $q=1200$'),(m_bu,PC[3],'P4: $q = 0$')]:
    if mask.any():
        #ax.fill_between(t[mask], 0, q[mask], color=col, alpha=0.35)
        ax.plot(t[mask], q[mask], color=col, lw=1.8, label=lbl)
for tv in [t_p1end, t_p2end, t_p3end]:
    ax.axvline(tv, color=GR, lw=1.2, ls='--', alpha=0.7)
ax.set_ylabel('$q$ [STB/day]')
ax.set_title('(a)',loc='left',weight='normal')
ax.legend(ncol=4, fontsize=9, framealpha=0.93)
ax.set_ylim([0, 1500])

# Panel (b): BHP
ax2 = axes[1]
for mask, col in [(m_p1,PC[0]),(m_p2,PC[1]),(m_p3,PC[2]),(m_bu,PC[3])]:
    if mask.any():
        ax2.semilogx(t[mask], bhp[mask], color=col, lw=2.0)
for tv in [t_p1end, t_p2end, t_p3end]:
    ax2.axvline(tv, color=GR, lw=1.2, ls='--', alpha=0.7)
ax2.set_ylabel('$p_{wf}$ [psia]')
ax2.set_title('(b)',loc='left',weight='normal')
ax2.set_ylim([3500, 4500])


# Panel (c): cumulative production
ax3 = axes[2]
for mask, col in [(m_p1,PC[0]),(m_p2,PC[1]),(m_p3,PC[2]),(m_bu,PC[3])]:
    if mask.any():
        ax3.semilogx(t[mask], Np[mask]/1e6, color=col, lw=2.0)
for tv in [t_p1end, t_p2end, t_p3end]:
    ax3.axvline(tv, color=GR, lw=1.2, ls='--', alpha=0.7)
ax3.set_xlabel('$t$ [hours]')
ax3.set_ylabel('$N_p$ [MMSTB]')
ax3.set_title('(c)',loc='left',weight='normal')
ax3.set_xlim([0.01, 1e4])
ax3.set_ylim([0, 4])
ax3.xaxis.set_minor_locator(NullLocator())
ax3.yaxis.set_minor_locator(NullLocator())

plt.tight_layout()
sf('fig1_rate_schedule.pdf')
plt.show()

In [ ]:
k    = P['k_mD']; phi_ = P['phi']; ct  = P['ct_psi1']; mu  = P['mu_cp']
rw   = P['rw_ft']; kh  = P['kh_mDft']; Bo  = P['Bo']; h_ft = P['h_ft']
q1   = P['q1_STBd']; q2 = P['q2_STBd']; q3 = P['q3_STBd']
dp_flat = P['dp_prime_psi']      # Bourdet flat for q1 reference
m_slope = P['m_slope_psi_cycle']
S_true  = P['S_skin']

def bourdet(dp_arr, t_arr, L=0.2):
    n,lnt,d = len(dp_arr),np.log(t_arr),np.zeros(len(dp_arr))
    for i in range(1,n-1):
        jl=i-1
        while jl>0   and lnt[i]-lnt[jl]<L: jl-=1
        jr=i+1
        while jr<n-1 and lnt[jr]-lnt[i]<L: jr+=1
        dL,dR = lnt[i]-lnt[jl], lnt[jr]-lnt[i]
        if dL>0 and dR>0:
            d[i]=((dp_arr[i]-dp_arr[jl])/dL*dR+(dp_arr[jr]-dp_arr[i])/dR*dL)/(dL+dR)
        elif dL>0: d[i]=(dp_arr[i]-dp_arr[jl])/dL
        elif dR>0: d[i]=(dp_arr[jr]-dp_arr[i])/dR
    d[0]=(dp_arr[1]-dp_arr[0])/(lnt[1]-lnt[0])
    d[-1]=(dp_arr[-1]-dp_arr[-2])/(lnt[-1]-lnt[-2])
    return d

# Rate-normalised pressure and material balance time
# Avoid division by zero during shut-in (q=0)
eps_q    = 1e-6
dp_norm  = dp / np.maximum(q, eps_q)         # Δp/q [psi/(STB/day)]
t_mb     = Np  / np.maximum(q, eps_q)         # t_mb [hours]

# Bourdet derivative of normalised pressure vs t_mb
prod_mask  = q > eps_q                         # producing timesteps only
dp_norm_p  = dp_norm[prod_mask]
t_mb_p     = t_mb[prod_mask]
sort_idx   = np.argsort(t_mb_p)
t_mb_s     = t_mb_p[sort_idx]
dp_norm_s  = dp_norm_p[sort_idx]
dpr_norm   = bourdet(dp_norm_s, t_mb_s, L=0.3)

# Superposition time (3-rate history before buildup)
# t_sup = sum_j (q_j - q_{j-1}) * log10(t_n - t_{j-1})
# Using field-unit MTR: Δp/q_last = m * t_sup + b
q_hist = np.array([0.0, q1, q2, q3])          # rate before each change
t_hist = np.array([0.0, 0.0, t_p1end, t_p2end]) # time of each change

t_sup = np.zeros(len(t))
for k_idx in range(len(t)):
    if q[k_idx] < eps_q: continue             # only during production
    s = 0.0
    for j in range(1, len(q_hist)):
        dt_j = t[k_idx] - t_hist[j]
        if dt_j > 0:
            s += (q_hist[j] - q_hist[j-1]) * np.log10(dt_j)
    t_sup[k_idx] = s

# Period 1 only: conventional Bourdet for regime identification
dpr_p1 = bourdet(np.clip(dp[m_p1],0,None), t[m_p1], L=0.2)
vs_p1  = (dpr_p1>1e-2) & (dp[m_p1]>1e-2)

print(f'Rate-norm range : {dp_norm[prod_mask].min():.4f} – '
      f'{dp_norm[prod_mask].max():.3f} psi/(STB/day)')
print(f't_mb range      : {t_mb_s[1]:.4f} – {t_mb_s[-1]:.1f} hr')
print(f'Bourdet flat(q1): {dp_flat:.4f} psi  (theory)')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))

log_lo=np.log10(1.0); log_hi=np.log10(1e4)
log_range=log_hi-log_lo
def xpos(tv): return (np.log10(np.clip(tv,1,1e4))-log_lo)/log_range
wbs_x=xpos(np.sqrt(1.0*t_wbs)); rf_x=xpos(np.sqrt(t_wbs*t_pss))
bdf_x=xpos(np.sqrt(t_pss*1e4))

ax=axes[0]
ax.semilogx(t[m_p1],bhp[m_p1],color='k',alpha=0.6,lw=2.0,
            label='Simulation Data  (Period 1)',zorder=4)
ax.axvline(t_wbs,color='green',lw=2,ls=':',alpha=0.8)
ax.axvline(t_pss,color='blue', lw=2,ls=':',alpha=0.8)
ax.axvspan(1,    t_wbs,color='C3',alpha=0.15)
ax.axvspan(t_wbs,t_pss,color='C2',alpha=0.12)
ax.axvspan(t_pss,1e4,  color='C4',alpha=0.15)
for xp,lbl in [(wbs_x,'WBS'),(rf_x,'RF'),(bdf_x,'BDF')]:
    ax.text(xp,0.70,lbl,transform=ax.transAxes,fontsize=10,
            color='black',ha='center',verticalalignment='top',
            bbox=dict(facecolor='white',alpha=0.7,edgecolor='none'))
ax.set_title('(a)',loc='left',weight='normal')
ax.set_xlabel('$t$ [hours]'); ax.set_ylabel('$p_{wf}$ [psia]')
ax.set_xlim([1,1e4])
ax.legend(loc='upper right',framealpha=0.93)

ax2=axes[1]
ax2.axvspan(0.1, t_wbs,color='C3',alpha=0.15)
ax2.axvspan(t_wbs,t_pss,color='C2',alpha=0.12)
ax2.axvspan(t_pss,1e4,  color='C4',alpha=0.15)
ax2.axvline(t_wbs,color='green',lw=2,ls=':',alpha=0.8)
ax2.axvline(t_pss,color='blue', lw=2,ls=':',alpha=0.8)
ax2.loglog(t[m_p1][vs_p1],dpr_p1[vs_p1],
           color='k',alpha=0.6,lw=2.0,label='Simulation Data  (Period 1)',zorder=4)
ax2.loglog([0.1,1e4],[dp_flat,dp_flat],'--',color='C1',lw=2.5,
           label=rf"$\Delta p'={dp_flat:.2f}$ psi (theory, $q_1$)")
for xp,lbl in [((np.log10(np.sqrt(0.1*t_wbs))-np.log10(0.1))/(np.log10(1e4)-np.log10(0.1)),'WBS'),
               ((np.log10(np.sqrt(t_wbs*t_pss))-np.log10(0.1))/(np.log10(1e4)-np.log10(0.1)),'RF'),
               ((np.log10(np.sqrt(t_pss*1e4))-np.log10(0.1))/(np.log10(1e4)-np.log10(0.1)),'BDF')]:
    ax2.text(xp,0.75,lbl,transform=ax2.transAxes,fontsize=10,
             color='black',ha='center',verticalalignment='top',
             bbox=dict(facecolor='white',alpha=0.7,edgecolor='none'))
ax2.set_title('(b)',loc='left',weight='normal')
ax2.set_xlabel('$t$ [hours]'); ax2.set_ylabel(r"$\Delta p'$ [psi]")
ax2.set_xlim([0.1,1e4]); ax2.set_ylim([1,1000])
ax2.legend(loc='lower right',framealpha=0.93,edgecolor='k')
plt.tight_layout()
plt.show()

In [ ]:
# ── Single panel: Rate-normalised pressure ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 2.5))

# Plot the flowing periods
for mask, col, lbl in [(m_p1, PC[0], f'P1  $q={q1:.0f}$ STB/day'),
                       (m_p2, PC[1], f'P2  $q={q2:.0f}$ STB/day'),
                       (m_p3, PC[2], f'P3  $q={q3:.0f}$ STB/day')]:
    
    vm = mask & prod_mask  # Ensure this only grabs flowing time
    
    if vm.any():
        ax.loglog(t_mb[vm], dp_norm[vm], '-',
                  color=col, lw=2.5, alpha=0.8, label=lbl)

# Uncomment if you want to show the theoretical plateau
# ax.axhline(dp_flat/q1, color='black', lw=1.5, ls='--', alpha=0.7,
#            label=rf"RF plateau = {dp_flat/q1:.4f} psi/(STB/day)")

ax.set_xlabel('$t_{mb} = N_p / q$  [hours]')
ax.set_ylabel(r'$\Delta p / q$  [psi/(STB/day)]')
#ax.set_title('Rate-Normalized Pressure (Blasingame)', loc='left', weight='normal')

ax.set_xlim([1e-2, 1e4])
ax.set_ylim([0.1, 1])

# Hide minor ticks for standard clean formatting
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

ax.legend(loc='lower right', fontsize=10, framealpha=0.93)

plt.tight_layout()
sf('fig3_vrate_rta_diagnostic_single.pdf')
plt.show()

In [ ]:
# =============================================================================
#  Combined Figure — (a) Rate schedule + BHP  |  (b) RTA diagnostic
#  Insert this cell immediately after the fig3 cell.
#  All variables are already defined; no new data loading required.
# =============================================================================


# ── Layout ────────────────────────────────────────────────────────────────────
# figsize=(8, 3.5): left half = panel (a) split into q (top) + p_wf (bottom)
#                   right half = panel (b) full-height RTA
# ─────────────────────────────────────────────────────────────────────────────

FS_LABEL  = 14   # axis labels, panel letters
FS_TICK   = 12   # tick labels
FS_INSIDE = 10   # legend, in-plot annotations

fig = plt.figure(figsize=(8, 3.5))

gs_outer = gridspec.GridSpec(
    1, 2, figure=fig,
    wspace=0.40,            # horizontal gap between (a) and (b)
    left=0.09, right=0.97,
    top=0.94, bottom=0.13,
)

# Left column: two stacked panels sharing x-axis
gs_left = gridspec.GridSpecFromSubplotSpec(
    2, 1,
    subplot_spec=gs_outer[0],
    hspace=0.1,            # tight vertical gap so shared x-axis ticks don't overlap
    height_ratios=[1, 1],
)

ax_q   = fig.add_subplot(gs_left[0])                      # top  — rate
ax_pwf = fig.add_subplot(gs_left[1], sharex=ax_q)         # bottom — BHP
ax_rta = fig.add_subplot(gs_outer[1])                      # right — RTA

# ─────────────────────────────────────────────────────────────────────────────
# Panel (a-top) — Rate history
# ─────────────────────────────────────────────────────────────────────────────
for mask, col, lbl in [
        (m_p1, PC[0], r'P1: $q=800$'),
        (m_p2, PC[1], r'P2: $q=400$'),
        (m_p3, PC[2], r'P3: $q=1200$'),
        (m_bu, PC[3], r'P4: $q=0$'),
]:
    if mask.any():
        ax_q.semilogx(t[mask], q[mask], color=col, lw=3, label=lbl)

for tv in [t_p1end, t_p2end, t_p3end]:
    ax_q.axvline(tv, color=GR, lw=1.0, ls='--', alpha=0.65)

ax_q.set_ylabel(r'$q$ [STB/day]',     fontsize=FS_LABEL)
ax_q.set_ylim([0, 1500])
ax_q.set_xlim([0.01, 1e4])
ax_q.tick_params(labelbottom=False, labelsize=FS_TICK)   # hide shared x-labels
ax_q.xaxis.set_minor_locator(NullLocator())
ax_q.yaxis.set_minor_locator(NullLocator())
ax_q.legend(
    ncol=2, fontsize=FS_INSIDE,
    framealpha=0.0, loc='lower left',
    handlelength=1.2, labelspacing=0.3,
)
ax_q.set_title('(a)', loc='left', fontsize=FS_LABEL, fontweight='normal', pad=3)

# ─────────────────────────────────────────────────────────────────────────────
# Panel (a-bottom) — BHP
# ─────────────────────────────────────────────────────────────────────────────
for mask, col in [
        (m_p1, PC[0]),
        (m_p2, PC[1]),
        (m_p3, PC[2]),
        (m_bu, PC[3]),
]:
    if mask.any():
        ax_pwf.semilogx(t[mask], bhp[mask], color=col, lw=3)

for tv in [t_p1end, t_p2end, t_p3end]:
    ax_pwf.axvline(tv, color=GR, lw=1.0, ls='--', alpha=0.65)

ax_pwf.set_xlabel(r'$t$ [hours]',       fontsize=FS_LABEL)
ax_pwf.set_ylabel(r'$p_{wf}$ [psia]',   fontsize=FS_LABEL)
ax_pwf.set_ylim([3500, 4500])
ax_pwf.tick_params(labelsize=FS_TICK)
ax_pwf.xaxis.set_minor_locator(NullLocator())
ax_pwf.yaxis.set_minor_locator(NullLocator())

# ─────────────────────────────────────────────────────────────────────────────
# Panel (b) — RTA: rate-normalised pressure  (fig3 content)
# ─────────────────────────────────────────────────────────────────────────────
for mask, col, lbl in [
        (m_p1, PC[0], fr'P1  $q={q1:.0f}$ STB/day'),
        (m_p2, PC[1], fr'P2  $q={q2:.0f}$ STB/day'),
        (m_p3, PC[2], fr'P3  $q={q3:.0f}$ STB/day'),
]:
    vm = mask & prod_mask
    if vm.any():
        ax_rta.loglog(t_mb[vm], dp_norm[vm], '-',
                      color=col, lw=3.0, alpha=0.85, label=lbl)

ax_rta.set_xlabel(r'$t_{mb} = N_p/q$  [hours]',          fontsize=FS_LABEL)
ax_rta.set_ylabel(r'$\Delta p / q$  [psi/(STB/day)]',    fontsize=FS_LABEL)
ax_rta.set_xlim([1e-2, 1e4])
ax_rta.set_ylim([0.1,  1.0])
ax_rta.tick_params(labelsize=FS_TICK)
ax_rta.xaxis.set_minor_locator(NullLocator())
ax_rta.yaxis.set_minor_locator(NullLocator())
ax_rta.legend(loc='lower right', fontsize=FS_INSIDE, framealpha=0.93)
ax_rta.set_title('(b)', loc='left', fontsize=FS_LABEL, fontweight='normal', pad=3)

# ─────────────────────────────────────────────────────────────────────────────
# Save
# ─────────────────────────────────────────────────────────────────────────────
sf('ch4_5_q_pwf_ndp_case4.pdf')
plt.show()


---
## 2 · Rate Schedule Visualisation

The four production periods are shown alongside the BHP and cumulative production. Rate changes at 1000 hr, 2500 hr and 4000 hr are visible as discontinuities in BHP — the simulator captures the transient response to each rate change.

---
## 3 · Rate-Transient Analysis (RTA) Diagnostics

### 3.1 Theory

For variable-rate production the direct Bourdet derivative is distorted by rate changes. RTA uses two diagnostic transformations:

**Rate-normalised pressure** (Blasingame):
$$\frac{\Delta p}{q} = \frac{p_i - p_{wf}(t)}{q(t)}$$

**Material balance time** (equivalent to constant-rate time):
$$t_{mb} = \frac{N_p(t)}{q(t)}$$

A log-log plot of $\Delta p/q$ vs $t_{mb}$ should overlay all rate periods onto a single diagnostic curve — the multi-rate analogue of the Bourdet plot. The same flow regime signatures apply: slope 0 = radial flow, slope 1 = BDF (pseudosteady state).

**Superposition time** (rigorous multi-rate):
$$t_{sup} = \sum_{j=1}^{n} (q_j - q_{j-1}) \log(t - t_{j-1})$$

A plot of $\Delta p/q_n$ vs $t_{sup}$ linearises during MTR.

In [ ]:
k    = P['k_mD']; phi_ = P['phi']; ct  = P['ct_psi1']; mu  = P['mu_cp']
rw   = P['rw_ft']; kh  = P['kh_mDft']; Bo  = P['Bo']; h_ft = P['h_ft']
q1   = P['q1_STBd']; q2 = P['q2_STBd']; q3 = P['q3_STBd']
dp_flat = P['dp_prime_psi']      # Bourdet flat for q1 reference
m_slope = P['m_slope_psi_cycle']
S_true  = P['S_skin']

def bourdet(dp_arr, t_arr, L=0.2):
    n,lnt,d = len(dp_arr),np.log(t_arr),np.zeros(len(dp_arr))
    for i in range(1,n-1):
        jl=i-1
        while jl>0   and lnt[i]-lnt[jl]<L: jl-=1
        jr=i+1
        while jr<n-1 and lnt[jr]-lnt[i]<L: jr+=1
        dL,dR = lnt[i]-lnt[jl], lnt[jr]-lnt[i]
        if dL>0 and dR>0:
            d[i]=((dp_arr[i]-dp_arr[jl])/dL*dR+(dp_arr[jr]-dp_arr[i])/dR*dL)/(dL+dR)
        elif dL>0: d[i]=(dp_arr[i]-dp_arr[jl])/dL
        elif dR>0: d[i]=(dp_arr[jr]-dp_arr[i])/dR
    d[0]=(dp_arr[1]-dp_arr[0])/(lnt[1]-lnt[0])
    d[-1]=(dp_arr[-1]-dp_arr[-2])/(lnt[-1]-lnt[-2])
    return d

# Rate-normalised pressure and material balance time
# Avoid division by zero during shut-in (q=0)
eps_q    = 1e-6
dp_norm  = dp / np.maximum(q, eps_q)         # Δp/q [psi/(STB/day)]
t_mb     = Np  / np.maximum(q, eps_q)         # t_mb [hours]

# Bourdet derivative of normalised pressure vs t_mb
prod_mask  = q > eps_q                         # producing timesteps only
dp_norm_p  = dp_norm[prod_mask]
t_mb_p     = t_mb[prod_mask]
sort_idx   = np.argsort(t_mb_p)
t_mb_s     = t_mb_p[sort_idx]
dp_norm_s  = dp_norm_p[sort_idx]
dpr_norm   = bourdet(dp_norm_s, t_mb_s, L=0.3)

# Superposition time (3-rate history before buildup)
# t_sup = sum_j (q_j - q_{j-1}) * log10(t_n - t_{j-1})
# Using field-unit MTR: Δp/q_last = m * t_sup + b
q_hist = np.array([0.0, q1, q2, q3])          # rate before each change
t_hist = np.array([0.0, 0.0, t_p1end, t_p2end]) # time of each change

t_sup = np.zeros(len(t))
for k_idx in range(len(t)):
    if q[k_idx] < eps_q: continue             # only during production
    s = 0.0
    for j in range(1, len(q_hist)):
        dt_j = t[k_idx] - t_hist[j]
        if dt_j > 0:
            s += (q_hist[j] - q_hist[j-1]) * np.log10(dt_j)
    t_sup[k_idx] = s

# Period 1 only: conventional Bourdet for regime identification
dpr_p1 = bourdet(np.clip(dp[m_p1],0,None), t[m_p1], L=0.2)
vs_p1  = (dpr_p1>1e-2) & (dp[m_p1]>1e-2)

print(f'Rate-norm range : {dp_norm[prod_mask].min():.4f} – '
      f'{dp_norm[prod_mask].max():.3f} psi/(STB/day)')
print(f't_mb range      : {t_mb_s[1]:.4f} – {t_mb_s[-1]:.1f} hr')
print(f'Bourdet flat(q1): {dp_flat:.4f} psi  (theory)')

### 3.2 Figure 2 — Period 1 PTA (Direct, same as Case 1)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))

log_lo=np.log10(1.0); log_hi=np.log10(1e4)
log_range=log_hi-log_lo
def xpos(tv): return (np.log10(np.clip(tv,1,1e4))-log_lo)/log_range
wbs_x=xpos(np.sqrt(1.0*t_wbs)); rf_x=xpos(np.sqrt(t_wbs*t_pss))
bdf_x=xpos(np.sqrt(t_pss*1e4))

ax=axes[0]
ax.semilogx(t[m_p1],bhp[m_p1],color='k',alpha=0.6,lw=2.0,
            label='Simulation Data  (Period 1)',zorder=4)
ax.axvline(t_wbs,color='green',lw=2,ls=':',alpha=0.8)
ax.axvline(t_pss,color='blue', lw=2,ls=':',alpha=0.8)
ax.axvspan(1,    t_wbs,color='C3',alpha=0.15)
ax.axvspan(t_wbs,t_pss,color='C2',alpha=0.12)
ax.axvspan(t_pss,1e4,  color='C4',alpha=0.15)
for xp,lbl in [(wbs_x,'WBS'),(rf_x,'RF'),(bdf_x,'BDF')]:
    ax.text(xp,0.70,lbl,transform=ax.transAxes,fontsize=10,
            color='black',ha='center',verticalalignment='top',
            bbox=dict(facecolor='white',alpha=0.7,edgecolor='none'))
ax.set_title('(a)',loc='left',weight='normal')
ax.set_xlabel('$t$ [hours]'); ax.set_ylabel('$p_{wf}$ [psia]')
ax.set_xlim([1,1e4])
ax.legend(loc='upper right',framealpha=0.93)

ax2=axes[1]
ax2.axvspan(0.1, t_wbs,color='C3',alpha=0.15)
ax2.axvspan(t_wbs,t_pss,color='C2',alpha=0.12)
ax2.axvspan(t_pss,1e4,  color='C4',alpha=0.15)
ax2.axvline(t_wbs,color='green',lw=2,ls=':',alpha=0.8)
ax2.axvline(t_pss,color='blue', lw=2,ls=':',alpha=0.8)
ax2.loglog(t[m_p1][vs_p1],dpr_p1[vs_p1],
           color='k',alpha=0.6,lw=2.0,label='Simulation Data  (Period 1)',zorder=4)
ax2.loglog([0.1,1e4],[dp_flat,dp_flat],'--',color='C1',lw=2.5,
           label=rf"$\Delta p'={dp_flat:.2f}$ psi (theory, $q_1$)")
for xp,lbl in [((np.log10(np.sqrt(0.1*t_wbs))-np.log10(0.1))/(np.log10(1e4)-np.log10(0.1)),'WBS'),
               ((np.log10(np.sqrt(t_wbs*t_pss))-np.log10(0.1))/(np.log10(1e4)-np.log10(0.1)),'RF'),
               ((np.log10(np.sqrt(t_pss*1e4))-np.log10(0.1))/(np.log10(1e4)-np.log10(0.1)),'BDF')]:
    ax2.text(xp,0.75,lbl,transform=ax2.transAxes,fontsize=10,
             color='black',ha='center',verticalalignment='top',
             bbox=dict(facecolor='white',alpha=0.7,edgecolor='none'))
ax2.set_title('(b)',loc='left',weight='normal')
ax2.set_xlabel('$t$ [hours]'); ax2.set_ylabel(r"$\Delta p'$ [psi]")
ax2.set_xlim([0.1,1e4]); ax2.set_ylim([1,1000])
ax2.legend(loc='lower right',framealpha=0.93,edgecolor='k')
plt.tight_layout()
plt.show()

### 3.3 Figure 3 — Rate-Normalised Diagnostic (All Periods)

### 3.4 Figure 4 — Full BHP History (All 4 Periods)

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))

for mask,col,lbl in [(m_p1,PC[0],f'P1: $q_1={q1:.0f}$ STB/day'),
                      (m_p2,PC[1],f'P2: $q_2={q2:.0f}$ STB/day'),
                      (m_p3,PC[2],f'P3: $q_3={q3:.0f}$ STB/day'),
                      (m_bu,PC[3],'P4: shut-in')]:
    if mask.any():
        ax.semilogx(t[mask],bhp[mask],color=col,lw=2.0,label=lbl)

for tv in [t_p1end,t_p2end,t_p3end]:
    ax.axvline(tv,color=GR,lw=1.5,ls='--',alpha=0.7)
ax.axvline(t_wbs,color='green',lw=1.5,ls=':',alpha=0.7)
ax.axvline(t_pss,color='blue', lw=1.5,ls=':',alpha=0.7)

ax.text(0.01,0.12,'WBS|RF',transform=ax.transAxes,fontsize=9,
        color='green',
        bbox=dict(facecolor='white',alpha=0.7,edgecolor='none'))
for tv,lbl in [(t_p1end,'P1|P2'),(t_p2end,'P2|P3'),(t_p3end,'P3|P4')]:
    xf_=(np.log10(tv)-np.log10(t[0]))/(np.log10(t[-1])-np.log10(t[0]))
    ax.text(xf_+0.005,0.90,lbl,transform=ax.transAxes,fontsize=8,
            color=GR,va='top')

ax.set_xlabel('$t$ [hours]'); ax.set_ylabel('$p_{wf}$ [psia]')
ax.set_title('Full BHP history — 4 production periods',
             loc='left',weight='normal')
ax.set_xlim([0.001,t[-1]])
ax.legend(ncol=2,framealpha=0.93,fontsize=9)
plt.tight_layout()
plt.show()

---
## 4 · Pressure Field Visualisation

One snapshot per period showing how the drawdown pattern evolves. The pressure field during Period 1 should be identical to Case 1 at the same elapsed time — validating the simulation setup. During Periods 2 and 3 the field responds to the rate changes. During the buildup (Period 4) the field recovers toward $p_i$.

In [ ]:
dX = P['p_init_psia'] - X
def snap_idx(tv): return int(np.argmin(np.abs(t-tv)))

# Guard: only include periods that actually have data
def safe_snap(mask, frac, fallback_mask):
    if mask.any():
        idx = min(int(len(t[mask]) * frac), len(t[mask]) - 1)  # clamp to last valid index
        return snap_idx(t[mask][idx])
    else:
        return snap_idx(t[fallback_mask][-1])

snaps = [
    (safe_snap(m_p1, 0.5,  m_p1),
     f'P1  $t={t[safe_snap(m_p1,0.5,m_p1)]:.0f}$ hr',   PC[0]),
    (safe_snap(m_p1, 1.0,  m_p1),
     f'P1 end $t={t[safe_snap(m_p1,1.0,m_p1)]:.0f}$ hr', PC[0]),
    (safe_snap(m_p2, 0.5,  m_p1),
     f'P2  $t={t[safe_snap(m_p2,0.5,m_p1)]:.0f}$ hr',   PC[1]),
    (safe_snap(m_p3, 0.25, m_p1),
     f'P3 early $t={t[safe_snap(m_p3,0.25,m_p1)]:.0f}$ hr', PC[2]),
    (safe_snap(m_p3, 0.75, m_p1),
     f'P3 late  $t={t[safe_snap(m_p3,0.75,m_p1)]:.0f}$ hr', PC[2]),
    (safe_snap(m_p3, 1.0,  m_p1),
     f'P3 end   $t={t[safe_snap(m_p3,1.0,m_p1)]:.0f}$ hr',  PC[2]),
]
# Replace last entry label if buildup data exists
if m_bu.any():
    #  snapshot 5% of the way through the buildup.
    bu_idx = safe_snap(m_bu, 0.05, m_p3)
    
    snaps[5] = (bu_idx,
                f'BU active $t={t[bu_idx]:.0f}$ hr', PC[3])

fig, axes = plt.subplots(2,3,figsize=(8,5))
for ax,(ki,ttl,col) in zip(axes.flat, snaps):
    F  = dX[:,ki].reshape(ny,nx)
    vm = max(F.max(),0.1)
    im = ax.imshow(F,origin='lower',cmap='hot_r',vmin=0,vmax=vm,aspect='equal')
    #ax.plot(nx//2,ny//2,'w+',ms=12,mew=2.5,zorder=10)
    for lp in [0.25,0.50,0.75]:
        lv=vm*lp
        if lv>0.05:
            ax.contour(F,levels=[lv],colors=['w'],linewidths=0.6,alpha=0.4)
    ax.set_title(ttl,fontsize=12,color=col,fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.add_patch(plt.Rectangle((0,0),1,1,fill=False,ec=col,lw=2.5,
                               transform=ax.transAxes,clip_on=False))
    cb=plt.colorbar(im,ax=ax,shrink=0.85)
    cb.set_ticks([0,vm/2,vm])
    cb.ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.0f'))
    cb.set_label(r'$\Delta p$ [psi]',fontsize=9)
   ## ax.text(0.97,0.03,f'max={vm:.0f}',transform=ax.transAxes,
            #fontsize=12,ha='right',va='bottom',color='white',family='monospace',
           # bbox=dict(fc='k',alpha=0.6,pad=1.5,ec='none'))
plt.tight_layout()
sf('ch4_10_p_field_case4.pdf')
plt.show()

---
## 5 · Proper Orthogonal Decomposition (POD)

The snapshot matrix spans all four production periods. The variable rate introduces additional temporal complexity relative to Case 1, so more POD modes are expected to capture the rate-change transients. Mode 1 still captures the dominant depletion; Mode 2 captures the first rate change at $t=1000$ hr; Mode 3 captures the second rate change at $t=2500$ hr.

In [ ]:
X_mean = X.mean(axis=1,keepdims=True)
Xc     = X - X_mean
U, sv, Vt = np.linalg.svd(Xc, full_matrices=False)

sn  = sv/sv[0]
cum = np.cumsum(sv**2)/np.sum(sv**2)*100

r99  = int(np.searchsorted(cum,99.0))+1
r999 = int(np.searchsorted(cum,99.9))+1
r_sel = max(r999,2)

print(f'Snapshot matrix  X : {X.shape}')
print(f'σ₁               : {sv[0]:.2f} psia')
print(f'σ₂/σ₁            : {sn[1]:.3e}  (vs Case 1 spectral gap)')
print(f'r for 99.0%%     : {r99}')
print(f'r for 99.9%%     : {r999}  ← selected r = {r_sel}')
for i in range(min(6,len(sv))):
    print(f'  Mode {i+1}  σ={sv[i]:.2f}  σᵢ/σ₁={sn[i]:.2e}  cumE={cum[i]:.4f}%')

### 5.1 Figure 6 — Singular Value Spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# --- Plot (a): Singular Value Decay ---
ax = axes[0]
ax.semilogy(range(1, 31), sn[:30], 'o', color=C1, ms=6.5, mec='k', mew=0.5)
ax.semilogy(range(1, 31), sn[:30], color='gray', alpha=0.3, lw=3, zorder=1)

ax.set_title('(a)', loc='left', fontweight='normal')
ax.set_xlabel('$r$')
ax.set_ylabel(r'$\sigma_r / \sigma_1$ [-]')

ax.set_xlim([0, 30])
ax.set_ylim([1e-10, 2])

# for idx, col in [(0, 'C3'), (1, 'C0'), (2, 'C5'), (3, 'C8')]:
#     ax.axvline(idx+1, color=col, ls=':', lw=1.5)
#     ax.text(idx+1, sn[min(idx, len(sn)-1)]*0.2, 
#             f'$i={idx+1}$', color=col, rotation=90, 
#             ha='right', va='center', fontweight='bold', 
#             bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

# --- Plot (b): Cumulative Energy ---
ax2 = axes[1]
ax2.plot(range(1, 31), cum[:30], 'o', color=C2, ms=6.5, mec='k', mew=0.5)
ax2.plot(range(1, 31), cum[:30], color='gray', alpha=0.3, lw=3, zorder=1)

ax2.set_title('(b)', loc='left', fontweight='normal')
ax2.set_xlabel('$r$')
ax2.set_ylabel(r'$E_r \ [\%]$')

ax2.set_xlim([0, 30])
# ax2.set_ylim([95, 100.5]) # Uncomment this if you want to force the standard Y-limits

x_positions = [22, 8, 22, 8]
va_positions = ['top', 'bottom', 'top', 'bottom']

# for ii, (idx, col) in enumerate([(0, 'C3'), (1, 'C0'), (2, 'C5'), (3, 'C8')]):
#     if idx < len(cum):
#         ax2.axhline(cum[idx], color=col, ls=':', lw=1.5)
#         ax2.text(x_positions[ii], cum[idx], 
#                  f'$r={idx+1} \ (E_r={cum[idx]:.2f}\\%)$', 
#                  color=col, ha='center', va=va_positions[ii], fontweight='bold', 
#                  bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

plt.tight_layout()
#sf('fig6_varrate_svd_spectrum.pdf')
sf('ch5_13_svd_rank_case4.pdf')
plt.show()

### 5.2 Figure 7 — POD Spatial Modes and Temporal Coefficients

In [ ]:
n_show = min(4, r_sel)  # Safely handle if fewer than 4 modes exist
fig, axes = plt.subplots(2, n_show, figsize=(9, 4.5))
if n_show == 1: axes = axes.reshape(2, 1)

mode_phys = ['Depletion\n(dominant)',
             'Rate change\nP1→P2 (t=1000 hr)',
             'Rate change\nP2→P3 (t=2500 hr)',
             'Higher-order\ncorrection']

for i in range(n_show):
    # --- spatial mode ---
    ax = axes[0, i]
    ui = U[:, i].reshape(ny, nx)
    vm = np.abs(ui).max()
    im = ax.imshow(ui, origin='lower', cmap='RdBu_r',
                   vmin=-vm, vmax=vm, aspect='equal')
    
    m_val = f'{sn[i]:.1e}'.split('e')[0]
    e_val = int(f'{sn[i]:.1e}'.split('e')[1])
    
    ax.set_title(
        f'${{u}}_{{{i+1}}}(\\mathbf{{x}})$\n'
        f'$(\\sigma_{{{i+1}}}/\\sigma_1={m_val}\\times10^{{{e_val}}})$\n', fontsize=12)
        
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    
    plt.colorbar(im, ax=ax, shrink=0.88, format='%.3f').set_label(
        rf'$u_{i+1}(\mathbf{{x}})\ [-]$', fontsize=12, rotation=90, labelpad=12)

    # --- temporal coefficient ---
    ax2 = axes[1, i]
    ai = sv[i] * Vt[i, :]
    ai_n = ai / np.abs(ai).max()
    
    col_seg = [(PC[0] if ti <= t_p1end else PC[1] if ti <= t_p2end
                else PC[2] if ti <= t_p3end else PC[3]) for ti in t]
                
    for j in range(len(t) - 1):
        ax2.semilogx(t[j:j+2], ai_n[j:j+2],
                     color=col_seg[j], lw=3, alpha=0.85)
                     
    # Vertical boundary lines
    for tv in [t_p1end, t_p2end, t_p3end]:
        ax2.axvline(tv, color=GR, lw=1.0, ls=':', alpha=0.7)
        
    ax2.axhline(0, color=GR, lw=1.0, alpha=0.4)
    ax2.set_xlabel('$t$ [hr]', fontsize=12)
    ax2.set_ylabel(f'$a_{i+1}(t)/|a_{i+1}|_{{max}}$', fontsize=12)
    ax2.set_xlim([0.01, 1e4])
    ax2.set_ylim([-1, 1])
    
# ── Figure-Level Legend for Rate Events ───────────────────────────────────
# Create custom legend lines for P1, P2, P3, P4 mapping to the PC colors
legend_handles = [
    mlines.Line2D([], [], color=PC[0], lw=3, label='P1'),
    mlines.Line2D([], [], color=PC[1], lw=3, label='P2'),
    mlines.Line2D([], [], color=PC[2], lw=3, label='P3'),
    mlines.Line2D([], [], color=PC[3], lw=3, label='P4')
]

# Place the legend at the bottom center, spanning 4 columns (single line)
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.02), 
           ncol=4, framealpha=0.0, fontsize=12)

# Adjust layout to reserve the bottom 8% of the figure canvas for the legend
plt.tight_layout(rect=[0, 0.01, 1, 1.02])

#sf('fig7_pod_modes_temporal.pdf')
sf('ch5_14_pod_modes_case4.pdf')
plt.show()

In [ ]:
GR = 'gray' if 'GR' not in locals() else GR

# =============================================================================
# SECTION 1: UNIFORM LOG-TIME RESAMPLING
# =============================================================================
# We use standard standard pressure drop from initial for SSA
dp_bhp   = P['p_init_psia'] - bhp
tau      = np.log(t)
tau_uni  = np.linspace(tau.min(), tau.max(), 500)
t_uni    = np.exp(tau_uni)

f_interp = interp1d(tau, dp_bhp, kind='cubic')
dp_uni   = f_interp(tau_uni)
dpr_uni  = np.gradient(dp_uni, tau_uni)

# =============================================================================
# SECTION 2: HANKEL EMBEDDING — built for both signals
# =============================================================================
def _build_hankel(signal, d):
    n = len(signal)
    H = np.zeros((d, n - d))
    for i in range(d):
        H[i, :] = signal[i : n - d + i]
    return H

d_h    = 20
W_h    = 5
k_rank = 3

H_dp  = _build_hankel(dp_uni,  d_h)   # from ΔP
H_dpr = _build_hankel(dpr_uni, d_h)   # from ΔP'

m_Hank = H_dp.shape[1]                # 500 - d_h = 480
t_Hank = t_uni[d_h:]                  # exactly m_Hank entries

# =============================================================================
# SECTION 3: SLIDING SSA — runs for both Hankel matrices
# =============================================================================
def _sliding_ssa(H_mat, t_Hank, W_h, k_rank):
    m      = H_mat.shape[1]
    n_h    = m - W_h
    t_str  = np.zeros(n_h)
    Rk     = np.zeros(n_h)
    H_ent  = np.zeros(n_h)

    for k_s in range(n_h):
        Xw          = H_mat[:, k_s : k_s + W_h]
        t_str[k_s]  = t_Hank[k_s + W_h // 2]

        try:
            _, S_w, _       = np.linalg.svd(Xw, full_matrices=False)
            energies        = S_w ** 2
            E_primary       = energies[0]

            # Multi-Rank Complexity
            Rk[k_s]         = np.sum(energies[1:k_rank]) / E_primary

            # Spectral Entropy
            e_trunc         = energies[energies > 1e-10 * E_primary]
            p_i             = e_trunc / e_trunc.sum()
            H_ent[k_s]      = -np.sum(p_i * np.log(p_i))

        except Exception:
            Rk[k_s]    = np.nan
            H_ent[k_s] = np.nan

    return t_str, Rk, H_ent

t_stream_dp,  Rk_dp,  H_dp_ent  = _sliding_ssa(H_dp,  t_Hank, W_h, k_rank)
t_stream_dpr, Rk_dpr, H_dpr_ent = _sliding_ssa(H_dpr, t_Hank, W_h, k_rank)

# =============================================================================
# SECTION 4: SMOOTHING
# =============================================================================
def _smooth(arr):
    valid = np.isfinite(arr)
    n     = valid.sum()
    win   = 31 if n > 31 else (n // 2) * 2 - 1
    out   = np.full(len(arr), np.nan)
    out[valid] = savgol_filter(arr[valid], window_length=win, polyorder=3)
    return out, valid

H_dp_smooth,  v_dp  = _smooth(H_dp_ent)
H_dpr_smooth, v_dpr = _smooth(H_dpr_ent)

# valid masks for Rk (raw, no smoothing)
vr_dp  = np.isfinite(Rk_dp)
vr_dpr = np.isfinite(Rk_dpr)

# =============================================================================
# SECTION 5: PLOTTING (With Log-Zoom Inset)
# =============================================================================
# Color palette
C_dp  = '#185FA5'   # blue  — ΔP
C_dpr = '#9b2a1a'   # red   — ΔP'

# Incorporate the Case 4 variable-rate events!
ref_events = [(t_wbs,   'green', r'$t_{wbs}$'),
              (t_pss,   'blue',  r'$t_{bdf}$'),
              (t_p1end, GR,      r'$P_2$ Start'),
              (t_p2end, GR,      r'$P_3$ Start'),
              (t_p3end, 'red',   r'Shut-in')]

fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
fig.subplots_adjust(hspace=0.08)

# --- (a) Multi-Rank Complexity Rk — twin y axes ---
ax1      = axes[0]
ax1_twin = ax1.twinx()

ax1.loglog     (t_stream_dp[vr_dp],   Rk_dp[vr_dp],   color=C_dp,  lw=3.0, label=r'$R_k$ ($\Delta P$)')
ax1_twin.loglog(t_stream_dpr[vr_dpr], Rk_dpr[vr_dpr], color=C_dpr, lw=3.0, label=r"$R_k$ ($\Delta P'$)", ls='--')

ax1.set_ylabel     (r'$R_k$  ($\Delta P$)',  color=C_dp)
ax1_twin.set_ylabel(r"$R_k$  ($\Delta P'$)", color=C_dpr)
ax1.tick_params(axis='y', labelcolor=C_dp)
ax1_twin.tick_params(axis='y', labelcolor=C_dpr)

ax1.set_ylim([1e-10, 1e2])
ax1_twin.set_ylim([1e-10, 1e2])
ax1.set_title('(a)', loc='left')
ax1.xaxis.set_minor_locator(NullLocator())
ax1.yaxis.set_minor_locator(NullLocator())
ax1_twin.yaxis.set_minor_locator(NullLocator())

lines  = ax1.get_lines() + ax1_twin.get_lines()
labels = [l.get_label() for l in lines]
#ax1.legend(lines, labels, fontsize=9, framealpha=0.9, loc='upper left')

# --- (b) Spectral Entropy H — twin y axes ---
ax2      = axes[1]
ax2_twin = ax2.twinx()

ax2.semilogx     (t_stream_dp[v_dp],   H_dp_smooth[v_dp],   color=C_dp,  lw=3.0, label=r'$H$ ($\Delta P$)')
ax2_twin.semilogx(t_stream_dpr[v_dpr], H_dpr_smooth[v_dpr], color=C_dpr, lw=3.0, label=r"$H$ ($\Delta P'$)", ls='--')

ax2.set_ylabel     (r'$H$  ($\Delta P$)',  color=C_dp)
ax2_twin.set_ylabel(r"$H$  ($\Delta P'$)", color=C_dpr)
ax2.tick_params(axis='y', labelcolor=C_dp)
ax2_twin.tick_params(axis='y', labelcolor=C_dpr)
ax2.set_xlabel(r'$t$  [hours]')
ax2.set_title('(b)', loc='left')

ax2.set_ylim([0, 0.3])
ax2_twin.set_ylim([0, 1.5])
ax2.xaxis.set_minor_locator(NullLocator())
ax2.yaxis.set_minor_locator(NullLocator())

lines2  = ax2.get_lines() + ax2_twin.get_lines()
labels2 = [l.get_label() for l in lines2]
#ax2.legend(lines2, labels2, fontsize=9, framealpha=0.9, loc='upper left')

# =============================================================================
# ── THE INSET FIGURE  ──
# =============================================================================
# Position: [x, y, width, height] relative to ax2 dimensions (0 to 1)
# Placed in the top-right corner to avoid covering the early-time physics
axins = ax2.inset_axes([0.32, 0.30, 0.33, 0.55])
axins_twin = axins.twinx()

# Re-plot the data inside the inset box
axins.semilogx(t_stream_dp[v_dp], H_dp_smooth[v_dp], color=C_dp, lw=3.0)
axins_twin.semilogx(t_stream_dpr[v_dpr], H_dpr_smooth[v_dpr], color=C_dpr, lw=3.0, ls='--')

# Re-draw the vertical reference lines inside the inset (only up to 2500)
for tv, col, _ in ref_events:
    if tv <= 2500:
        axins.axvline(tv, color=col, lw=1.0, ls=':', alpha=0.85)

# Set the limits specifically for the zoomed box
axins.set_xlim([0.01, 10000])
axins.set_ylim([0, 0.000005])
axins_twin.set_ylim([0, 0.0005])

# Make the inset text a bit smaller and hide minor ticks
axins.tick_params(axis='both', which='major')
axins.tick_params(axis='y', labelcolor=C_dp)               # Colored inset y-axis (blue)
axins_twin.tick_params(axis='both', which='major')
axins_twin.tick_params(axis='y', labelcolor=C_dpr)         # Colored inset twin y-axis (red)

axins.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.6f'))      
axins_twin.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f')) 
bbox_props = dict(facecolor='white', edgecolor='none', alpha=0.8, pad=0.5)
for label in axins.get_yticklabels() + axins_twin.get_yticklabels():
    label.set_bbox(bbox_props)
axins.xaxis.set_minor_locator(NullLocator())
axins.yaxis.set_minor_locator(NullLocator())
axins_twin.yaxis.set_minor_locator(NullLocator())
# =============================================================================

# --- Shared decorations ---
label_y_pos = [0.92, 0.92, 0.92, 0.22, 0.92] 

for ax_s in [ax1, ax2]:
    for tv, col, _ in ref_events:
        ax_s.axvline(tv, color=col, lw=1.5, ls=':', alpha=0.85)

for (tv, col, lbl), y_pos in zip(ref_events, label_y_pos):
    ax1.text(tv, y_pos, lbl,
             transform  = ax1.get_xaxis_transform(),
             fontsize=10, ha='center', color=col, fontweight='bold',
             bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1.0))

ax1.set_xlim([0.01, 1e4])

plt.tight_layout()
sf('ch6_4_ssa_case4.pdf')
plt.show()

---
## 8 · Reduced-Order Model (ROM) Reconstruction

Same Peaceman correction as Case 1 (homogeneous, $k=20$ mD).

The ROM is trained on the full multi-rate snapshot matrix and must reconstruct BHP across all four production periods — a stronger test than Cases 1–3 because the temporal dynamics are richer.

In [ ]:
r_eq_ft      = 0.2 * P['dx_ft']
kh_well_cell = P['k_well_cell_mD'] * P['h_ft']

# Time-varying Peaceman correction — uses actual rate at each timestep
# q array is already loaded from well_data.mat
peaceman_const   = (141.2 * P['mu_cp'] * P['Bo'] / kh_well_cell
                    * (np.log(r_eq_ft/P['rw_ft']) + P['S_skin']))
dp_correction_arr = peaceman_const * q          # shape (m,) — one value per step

# scalar reference (q1) for printing only
dp_correction_ref = peaceman_const * P['q1_STBd']

# ROM at selected rank
X_rom = U[:,:r_sel]@np.diag(sv[:r_sel])@Vt[:r_sel,:]+X_mean
eps_t = np.linalg.norm(X-X_rom,axis=0)/np.linalg.norm(X,axis=0)*100
eps_g = np.linalg.norm(X-X_rom,'fro')/np.linalg.norm(X,'fro')*100

print(f'Peaceman const        : {peaceman_const:.4f} psia/(STB/day)')
print(f'dp_correction (q1=800): {dp_correction_ref:.1f} psia')
print(f'dp_correction (q2=400): {peaceman_const*P["q2_STBd"]:.1f} psia')
print(f'dp_correction (q3=1200): {peaceman_const*P["q3_STBd"]:.1f} psia')
print(f'ROM rank r             : {r_sel}')
print(f'Global ε_G             : {eps_g:.4f}%')
print(f'Max step error         : {eps_t.max():.4f}%')

### 8.1 Figure 11 — ROM vs Simulator: BHP and Bourdet (Period 1 focus)

### 8.2 Figure 12 — ROM BHP over Full Variable-Rate History

In [ ]:
# ── Use Standard Colors/Styles ────────────────────────────────────────────
ranks  = [3, 5, 10]
colors = ['orange', 'purple', 'red']
styles = ['--', '-.', ':']

t_lo, t_hi = 0.01, 1e4
t_mask_full = (t >= t_lo)

fig, ax = plt.subplots(figsize=(8, 3))

# ── Simulator Plot ────────────────────────────────────────────────────────
ax.semilogx(t[t_mask_full], bhp_sim[t_mask_full],
            color='blue', alpha=0.6, lw=2.0, label='Simulator', zorder=4)

# ── ROM Plot (with all 4 errors inside legend) ────────────────────────────
for r, col, ls in zip(ranks, colors, styles):
    d = rom_data[r]
    
    # Safely format P2 and P3 errors (in case of NaNs, though they exist here)
    err_p2 = f'{d["el_p2"]:.3f}\\%' if np.isfinite(d["el_p2"]) else '\\text{—}'
    err_p3 = f'{d["el_p3"]:.3f}\\%' if np.isfinite(d["el_p3"]) else '\\text{—}'
    
    # Combine all four errors
    lbl = (rf'ROM $r={r}$  '
           rf'($\varepsilon_G={d["eg"]:.3f}\%$, '
           rf'$\varepsilon_{{w,P1}}={d["el"]:.3f}\%$, '
           rf'$\varepsilon_{{w,P2}}={err_p2}$, '
           rf'$\varepsilon_{{w,P3}}={err_p3}$)')
           
    ax.semilogx(t[t_mask_full], d['bhp_wf'][t_mask_full],
                color=col, lw=2.0, ls=ls, alpha=0.90, label=lbl)

# ── Period boundaries (No Shading) ────────────────────────────────────────
for tv in [t_p1end, t_p2end, t_p3end]:
    ax.axvline(tv, color=GR, lw=1.5, ls='--', alpha=0.7)

# ── Period labels x-positions (computed from log-axis fractions) ──────────
log_lo    = np.log10(t_lo)
log_range = np.log10(t_hi) - log_lo

def xpos(t_val):
    return (np.log10(np.clip(t_val, t_lo, t_hi)) - log_lo) / log_range

# Use geometric midpoints for text placement
p1_x = xpos(np.sqrt(1.0 * t_p1end))
p2_x = xpos(np.sqrt(t_p1end * t_p2end))
p3_x = xpos(np.sqrt(t_p2end * t_p3end))
p4_x = xpos(np.sqrt(t_p3end * t_p3end*2))

for xp, lbl in [(p1_x, 'P1'), (p2_x, 'P2'), (p3_x, 'P3'), (p4_x, 'P4')]:
    ax.text(xp, 0.95, lbl, transform=ax.transAxes, fontsize=10,
            color=GR, ha='center', verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# ── Formatting ────────────────────────────────────────────────────────────
#ax.set_title('Full History ROM Comparison', loc='left', weight='normal')
ax.set_xlabel('$t$ [hours]')
ax.set_ylabel('$p_{wf}$ [psia]')
ax.set_xlim([t_lo, t_hi])
ax.set_ylim([3400, 4600])
# Apply standard tick cleanup
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# Adjusted legend font size slightly to fit the longer text
ax.legend(loc='lower left', framealpha=0.93, fontsize=9)
plt.tight_layout()
sf('fig12_rom_full_history.pdf')
plt.show()

# ── Printed summary ───────────────────────────────────────────────────────
print(f'\n{"r":>4}  {"ε_G":>10}  {"ε_well P1":>12}  '
      f'{"ε_well P2":>12}  {"ε_well P3":>12}')
print('─'*55)
for r in ranks:
    d = rom_data[r]
    p2_str = f'{d["el_p2"]:>11.3f}%' if np.isfinite(d['el_p2']) else f'{"—":>12}'
    p3_str = f'{d["el_p3"]:>11.3f}%' if np.isfinite(d['el_p3']) else f'{"—":>12}'
    print(f'{r:4d}  {d["eg"]:>10.4f}%  {d["el"]:>11.3f}%  {p2_str}  {p3_str}')

In [ ]:
# ── Colors ────────────────────────────────────────────────────────────────
# Period colors: blue, yellow, green, cyan  (consistent across both panels)
PC = ['#1565c0', '#d4a017', '#1b7837', '#0097a7']

# POD rank colors: magenta replaces orange for better contrast vs yellow/cyan
ranks  = [3, 5, 10]
colors = ['magenta', 'purple', 'red']
styles = ['--', '-.', ':']

t_lo, t_hi = 0.01, 1e4
t_mask_full = (t >= t_lo)

# ── Consistent simulator RNP ──────────────────────────────────────────────
dp_sim_rta  = P['p_init_psia'] - bhp_sim
dp_norm_sim = dp_sim_rta / np.maximum(q, eps_q)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# ══════════════════════════════════════════════════════════════════════════
# ── Panel (a): BHP — simulator color-coded by period ─────────────────────
ax = axes[0]

# Simulator: one segment per period, each with its period colour
for mask, col, lbl in [
        (m_p1, PC[0], f'Sim.\\ P1 ($q={q1:.0f}$)'),
        (m_p2, PC[1], f'Sim.\\ P2 ($q={q2:.0f}$)'),
        (m_p3, PC[2], f'Sim.\\ P3 ($q={q3:.0f}$)'),
        (m_bu, PC[3], 'Sim.\\ P4 (BU)'),
]:
    vm = mask & t_mask_full
    if vm.any():
        ax.semilogx(t[vm], bhp_sim[vm],
                    color=col, lw=3, alpha=0.85, label=lbl, zorder=4)

# POD reconstructions
for r, col, ls in zip(ranks, colors, styles):
    d = rom_data[r]
    err_p2 = f'{d["el_p2"]:.3f}\\%' if np.isfinite(d["el_p2"]) else '\\text{—}'
    err_p3 = f'{d["el_p3"]:.3f}\\%' if np.isfinite(d["el_p3"]) else '\\text{—}'
    lbl = (rf'POD  $r={r}$  '
           rf'($\varepsilon_G={d["eg"]:.3f}\%$, '
           rf'$\varepsilon_{{w,P1}}={d["el"]:.3f}\%$, '
           rf'$\varepsilon_{{w,P2}}={err_p2}$, '
           rf'$\varepsilon_{{w,P3}}={err_p3}$)')
    ax.semilogx(t[t_mask_full], d['bhp_wf'][t_mask_full],
                color=col, lw=3.0, ls=ls, alpha=0.80, label=lbl)

# Color-coded vlines and period labels
for tv, col in [(t_p1end, PC[0]), (t_p2end, PC[1]), (t_p3end, PC[2])]:
    ax.axvline(tv, color=col, lw=1.5, ls=':', alpha=0.8)

log_lo    = np.log10(t_lo)
log_range = np.log10(t_hi) - log_lo
def xpos(t_val):
    return (np.log10(np.clip(t_val, t_lo, t_hi)) - log_lo) / log_range

p1_x = xpos(np.sqrt(1.0 * t_p1end))
p2_x = xpos(np.sqrt(t_p1end * t_p2end))
p3_x = xpos(np.sqrt(t_p2end * t_p3end))
p4_x = xpos(np.sqrt(t_p3end * t_p3end * 2))

for lbl, xp, col, yp, va in [
        ('P1', p1_x, PC[0], 0.97, 'top'),
        ('P2', p2_x, PC[1], 0.03, 'bottom'),
        ('P3', p3_x, PC[2], 0.97, 'top'),
        ('P4', p4_x, PC[3], 0.03, 'bottom'),
]:
    ax.text(xp, yp, lbl, transform=ax.transAxes, fontsize=12,
            color=col, ha='center', va=va,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

ax.set_title('(a)', loc='left', weight='normal', fontsize=14)
ax.set_xlabel('$t$ [hours]', fontsize=14)
ax.set_ylabel('$p_{wf}$ [psia]', fontsize=14)
ax.set_xlim([t_lo, t_hi])
ax.set_ylim([3250, 4500])
ax.xaxis.set_minor_locator(NullLocator())
ax.yaxis.set_minor_locator(NullLocator())

# Panel (a) legend: simulator period entries only
sim_h, sim_l = zip(*[(h, l) for h, l in zip(*ax.get_legend_handles_labels())
                     if 'Sim.' in l])
ax.legend(list(sim_h), list(sim_l), loc='lower left',
          framealpha=0.0, edgecolor='black', fontsize=12)

# ══════════════════════════════════════════════════════════════════════════
# ── Panel (b): RTA Rate-Normalized Pressure ───────────────────────────────
ax2 = axes[1]

# Simulator RNP segments
for mask, col, lbl in [
        (m_p1, PC[0], fr'Sim.\ P1 ($q={q1:.0f}$)'),
        (m_p2, PC[1], fr'Sim.\ P2 ($q={q2:.0f}$)'),
        (m_p3, PC[2], fr'Sim.\ P3 ($q={q3:.0f}$)'),
]:
    vm = mask & prod_mask
    if vm.any():
        ax2.loglog(t_mb[vm], dp_norm_sim[vm], '-',
                   color=col, lw=3, alpha=0.85, label=lbl, zorder=4)

# POD RNP segments (no label — covered by shared bottom legend)
for r, col, ls in zip(ranks, colors, styles):
    d = rom_data[r]
    dp_r = P['p_init_psia'] - d['bhp_wf']
    for mask in [m_p1, m_p2, m_p3]:
        vm = mask & prod_mask
        if vm.any():
            ax2.loglog(t_mb[vm], dp_r[vm] / np.maximum(q[vm], eps_q),
                       ls=ls, color=col, lw=3.0, alpha=0.80)

ax2.set_title('(b)', loc='left', weight='normal', fontsize=14)
ax2.set_xlabel(r'$t_{mb} = N_p/q$  [hours]', fontsize=14)
ax2.set_ylabel(r'$\Delta p / q$  [psi/(STB/d)]', fontsize=14)
ax2.set_xlim([1e-2, 1e4])
ax2.set_ylim([0.1, 1.0])
ax2.xaxis.set_minor_locator(NullLocator())
ax2.yaxis.set_minor_locator(NullLocator())

# Panel (b) legend: Simulator P1/P2/P3 only
ax2.legend(loc='lower right', framealpha=0.00, edgecolor='black', fontsize=12)

# ══════════════════════════════════════════════════════════════════════════
# ── Bottom legend: POD ranks only (shared across both panels) ─────────────
pod_handles, pod_labels = zip(*[(h, l) for h, l in zip(*ax.get_legend_handles_labels())
                                if 'POD' in l])
fig.legend(list(pod_handles), list(pod_labels),
           loc='upper center', bbox_to_anchor=(0.5, 0.01),
           ncol=1, framealpha=0.0, edgecolor='black', fontsize=12)

plt.tight_layout(rect=[0.0, 0.02, 1.0, 1.02])
#sf('fig12_pod_full_history_rta.pdf')
sf('ch5_15_pwf_dnp_recon_case4.pdf')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# §  Pressure Field Reconstruction Check — Case 4
#
# Figure A  (fig_recon_rank_P1RF.pdf)
#   n_ranks × 3  —  Ground Truth | POD r | Residual
#   Fixed snapshot at P1 radial flow: same layout as Cases 1-3.
#
# Figure B  (fig_recon_multiperiod.pdf)
#   4_periods × 3  —  one row per production period, r = r_sel
#   Case-4-specific: shows how POD handles the evolving drawdown
#   across each rate change and the buildup.
# ══════════════════════════════════════════════════════════════════════════

# ── Representative snapshot times ─────────────────────────────────────────
# P1: geometric midpoint of radial flow window (matches Cases 1-3 reference)
t_rf_snap = np.sqrt(t_wbs * t_pss)
# P2, P3: geometric midpoints of each flowing period
t_p2_snap = np.sqrt(t_p1end * t_p2end)
t_p3_snap = np.sqrt(t_p2end * t_p3end)
# P4 (buildup): 20 % into the BU window — early, active pressure recovery
t_bu_snap = t_shut + 0.20 * (t[-1] - t_shut)

def _ki(tv):
    return int(np.argmin(np.abs(t - tv)))

ki_rf = _ki(t_rf_snap)
ki_p2 = _ki(t_p2_snap)
ki_p3 = _ki(t_p3_snap)
ki_bu = _ki(t_bu_snap) if m_bu.any() else None

# Period descriptor: (snapshot_index, row_label, period_colour, valid_mask)
period_snaps = [
    (ki_rf, rf'P1 RF ($t={t[ki_rf]:.1f}$ hr)',  PC[0], m_p1),
    (ki_p2, rf'P2 ($t={t[ki_p2]:.0f}$ hr)',   PC[1], m_p2),
    (ki_p3, rf'P3 ($t={t[ki_p3]:.0f}$ hr)',   PC[2], m_p3),
]
if ki_bu is not None:
    period_snaps.append(
        (ki_bu, rf'P4 BU ($t={t[ki_bu]:.0f}$ hr)', PC[3], m_bu))

print(f'Snapshot times  →  P1-RF: {t[ki_rf]:.1f} hr | '
      f'P2: {t[ki_p2]:.0f} hr | P3: {t[ki_p3]:.0f} hr'
      + (f' | P4-BU: {t[ki_bu]:.0f} hr' if ki_bu is not None else ''))

# ══════════════════════════════════════════════════════════════════════════
# FIGURE A — rank comparison at P1 RF (consistent with Cases 1-3)
# ══════════════════════════════════════════════════════════════════════════
ranks  = [3, 5, 10]
n_ranks = len(ranks)

ki    = ki_rf
F_gt  = dX[:, ki].reshape(ny, nx)
vm    = max(F_gt.max(), 0.1)

# Standardized figure size
fig_a, axes_a = plt.subplots(n_ranks, 3, figsize=(8, 6))

for ri, r in enumerate(ranks):
    X_r      = U[:, :r] @ np.diag(sv[:r]) @ Vt[:r, :] + X_mean
    dX_r     = (P['p_init_psia'] - X_r)[:, ki].reshape(ny, nx)
    diff     = F_gt - dX_r
    eps_step = (np.linalg.norm(X[:, ki] - X_r[:, ki]) / 
                np.linalg.norm(X[:, ki]) * 100)
    vd       = max(np.abs(diff).max(), 0.01)

    ax_gt  = axes_a[ri, 0]
    ax_pod = axes_a[ri, 1]
    ax_res = axes_a[ri, 2]

    # Ground truth — shown once in top-left, blanked otherwise
    if ri == 0:
        im1 = ax_gt.imshow(F_gt, origin='lower', cmap='hot_r', vmin=0, vmax=vm)
        ax_gt.set_title(rf'Ground Truth' + '\n' + rf'$t={t[ki]:.1f}$ hr (P1 RF)', 
                        fontweight='normal', fontsize=12)
        fig_a.colorbar(im1, ax=ax_gt, shrink=0.7).set_label(r'$\Delta p$ [psi]', size=15)
    else:
        ax_gt.axis('off')

    # POD reconstruction
    im2 = ax_pod.imshow(dX_r, origin='lower', cmap='hot_r', vmin=0, vmax=vm)
    ax_pod.set_title(rf'POD ($r={r}$)' + '\n' + rf'$\varepsilon_G={eps_step:.4f}\%$', 
                     fontweight='normal', fontsize=12)
    fig_a.colorbar(im2, ax=ax_pod, shrink=0.7).set_label(r'$\Delta p$ [psi]', size=15)

    # Residual
    im3 = ax_res.imshow(diff, origin='lower', cmap='seismic', vmin=-vd, vmax=vd)
    ax_res.set_title(rf'Residual ($r={r}$)' + '\n' + rf'Max $\Delta$: {vd:.2f} psi', 
                     fontweight='normal', fontsize=12)
    fig_a.colorbar(im3, ax=ax_res, shrink=0.7).set_label(r'$\Delta p$ [psi]', size=15)

    for ax_obj in [ax_gt, ax_pod, ax_res]:
        if ax_obj.axison:
            ax_obj.set_xticks([]); ax_obj.set_yticks([])

fig_a.tight_layout()
#sf('fig_recon_rank_P1RF.pdf')
sf('ch5_16a_p_field_recon_case4.pdf')
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# FIGURE B — multi-period reconstruction at r_sel (case-4-specific)
# ══════════════════════════════════════════════════════════════════════════
X_rsel = U[:, :r_sel] @ np.diag(sv[:r_sel]) @ Vt[:r_sel, :] + X_mean
n_per  = len(period_snaps)

# Scaled layout to accommodate the number of periods
fig_b, axes_b = plt.subplots(n_per, 3, figsize=(8, 2.0 * n_per))

for pi, (ki, row_lbl, col, _mask) in enumerate(period_snaps):
    F_gt     = dX[:, ki].reshape(ny, nx)
    dX_r     = (P['p_init_psia'] - X_rsel)[:, ki].reshape(ny, nx)
    diff     = F_gt - dX_r
    eps_step = (np.linalg.norm(X[:, ki] - X_rsel[:, ki]) / 
                np.linalg.norm(X[:, ki]) * 100)
    vd       = max(np.abs(diff).max(), 0.01)
    vm_p     = max(F_gt.max(), 0.1)   # per-period vmax

    ax_gt  = axes_b[pi, 0]
    ax_pod = axes_b[pi, 1]
    ax_res = axes_b[pi, 2]

    # Row label colour-coded by period
    im1 = ax_gt.imshow(F_gt, origin='lower', cmap='hot_r', vmin=0, vmax=vm_p)
    ax_gt.set_title(rf'Ground Truth' + '\n' + row_lbl, 
                    fontweight='normal', fontsize=12, color=col)
    fig_b.colorbar(im1, ax=ax_gt, shrink=0.7).set_label(r'$\Delta p$ [psi]', size=15)

    im2 = ax_pod.imshow(dX_r, origin='lower', cmap='hot_r', vmin=0, vmax=vm_p)
    ax_pod.set_title(rf'POD ($r={r_sel}$)' + '\n' + rf'$\varepsilon_G={eps_step:.4f}\%$', 
                     fontweight='normal', fontsize=12, color=col)
    fig_b.colorbar(im2, ax=ax_pod, shrink=0.7).set_label(r'$\Delta p$ [psi]', size=15)

    im3 = ax_res.imshow(diff, origin='lower', cmap='seismic', vmin=-vd, vmax=vd)
    ax_res.set_title(rf'Residual ($r={r_sel}$)' + '\n' + rf'Max $\Delta$: {vd:.2f} psi', 
                     fontweight='normal', fontsize=12, color=col)
    fig_b.colorbar(im3, ax=ax_res, shrink=0.7).set_label(r'$\Delta p$ [psi]', size=15)

    for ax_obj in [ax_gt, ax_pod, ax_res]:
        if ax_obj.axison:
            ax_obj.set_xticks([]); ax_obj.set_yticks([])

fig_b.tight_layout()
#sf('fig_recon_multiperiod.pdf')
sf('ch5_16b_p_field_recon_case4.pdf')
plt.show()

---
## 9 · Reservoir Characterisation

Characterisation uses Period 1 data only (same approach as Case 1), providing a direct comparison. The ROM-derived estimates are compared against the simulator and the Case 1 true values.

In [ ]:
def characterise(dp_arr, t_arr, mtr_m, vs_m, label):
    dpr_arr  = bourdet(np.clip(dp_arr,0,None), t_arr, L=0.2)
    flat_val = np.median(dpr_arr[mtr_m&vs_m&(dpr_arr>0)]) if (mtr_m&vs_m).any() else np.nan
    kh_b     = 70.6*q1*mu*Bo/flat_val if np.isfinite(flat_val) else np.nan
    k_b      = kh_b/h_ft if np.isfinite(kh_b) else np.nan
    cf_r     = np.polyfit(np.log10(t_arr[mtr_m]), dp_arr[mtr_m], 1) if mtr_m.sum()>2 else [np.nan,np.nan]
    kh_slope = 162.6*q1*mu*Bo/cf_r[0] if np.isfinite(cf_r[0]) and cf_r[0]>0 else np.nan
    dp_1hr   = float(np.interp(1.0, t_arr, dp_arr))
    S_est    = 1.1513*(dp_1hr/cf_r[0]-np.log10(k/(phi_*mu*ct*rw**2))+3.2275) if np.isfinite(kh_slope) else np.nan
    return dict(label=label, flat=flat_val, kh_b=kh_b, k_b=k_b,
                kh_s=kh_slope, S=S_est)

phi_=P['phi']; ct=P['ct_psi1']

mtr_p1 = m_rf   # RF mask in Period 1
vs_p1_  = (dpr_p1>1e-2)&(dp[m_p1]>1e-2)

res_sim = characterise(dp[m_p1], t[m_p1], mtr_p1[m_p1], vs_p1_,'Simulator')
results = [res_sim]
for r in ranks:
    X_r      = U[:,:r]@np.diag(sv[:r])@Vt[:r,:]+X_mean
    bhp_r_wf = X_r[wc,:]-dp_correction_arr
    dp_r_p1  = P['p_init_psia']-bhp_r_wf
    dpr_r_p1 = bourdet(np.clip(dp_r_p1,0,None),t,L=0.2)
    vs_r_p1  = mtr_p1&(dpr_r_p1>1e-2)&(dp_r_p1>1e-2)
    results.append(characterise(dp_r_p1[m_p1],t[m_p1],
                                mtr_p1[m_p1],vs_r_p1[m_p1],f'ROM r={r}'))

W=14
print('='*85)
print(' RESERVOIR CHARACTERISATION — Case 4 (Period 1, q=800 STB/day)')
print('='*85)
print(f'  True: k={k:.0f} mD  kh={kh:.0f} mD-ft  S={S_true:.0f}')
hdr=f'{"":22} {"True":>{W}} '+''.join([f'{r["label"]:>{W}} ' for r in results])
print(); print(hdr); print('-'*85)

def row(lbl,key,fmt,ref):
    line=f'{lbl:<22} {ref:>{W}{fmt}} '
    for res in results:
        v=res[key]
        line+=f'{v:>{W}{fmt}} ' if np.isfinite(v) else f'{"—":>{W}} '
    print(line)

row('Δp\' flat [psi]',   'flat', '.4f', dp_flat)
row('kh (Bourdet) [mDft]','kh_b', '.2f', kh)
row('k [mD]',             'k_b',  '.3f', k)
row('kh (slope)  [mDft]', 'kh_s', '.2f', kh)
row('Skin S [-]',         'S',    '.4f', S_true)
print('='*85)

---
## 10 · Summary

In [ ]:
import json as js
summary=dict(
    case='case4_variable_rate',
    k_mD=float(k), kh=float(kh), S=float(S_true),
    periods=[dict(q=q1,t_start=0,t_end=t_p1end),
             dict(q=q2,t_start=t_p1end,t_end=t_p2end),
             dict(q=q3,t_start=t_p2end,t_end=t_p3end),
             dict(q=0, t_start=t_p3end,t_end=float(t[-1]))],
    r_sel=int(r_sel), r999=int(r999),
    sigma1=float(sv[0]), sigma2_ratio=float(sn[1]),

    eps_global=float(eps_g),

    dp_flat_theory=float(dp_flat),
    dp_flat_sim=float(res_sim['flat']),
    kh_sim=float(res_sim['kh_b']),
)
out_file=PROJECT/'data'/'processed'/'case4_summary.json'
with open(out_file,'w') as f: js.dump(summary,f,indent=2)
print('── SUMMARY ──────────────────────────────────')
for k_,v_ in summary.items(): print(f'  {k_:<30}: {v_}')
print(f'Saved: {out_file}')